# Exploração da Ingestão — Membro 3

Notebook para rodar os extractors de `src/extractors/` por partes e inspecionar os resultados antes de plugar nas DAGs do Airflow.

- **Seção 1**: API REST do Ceará Transparente (contratos)
- **Seção 2**: PostgreSQL de origem (`empenhos`, `ordem_bancaria_orcamentaria`, `unidade_gestora`)

> Antes de rodar: copie `../.env.example` para `../.env` e ajuste `SOURCE_POSTGRES_URL` com as credenciais reais da infra do time. As células da Seção 1 (API pública) funcionam sem configuração adicional.

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from dotenv import load_dotenv
load_dotenv(project_root / ".env")

import pandas as pd
print(f"Project root: {project_root}")

## 1. API REST — Ceará Transparente (contratos)

### 1.1 Sanity check — uma única página

Antes de iterar tudo, vale olhar a resposta bruta de uma página só para conferir o formato (`summary.total_pages`, campos de `data`).

In [ ]:
import requests
from src.extractors import api_extractor

DATA_INICIO = "2026-01-01"
DATA_FIM = "2026-01-31"

session = requests.Session()
primeira_pagina = api_extractor._request_page(session, page=1, data_assinatura_inicio=DATA_INICIO, data_assinatura_fim=DATA_FIM)

print("Chaves do payload:", list(primeira_pagina.keys()))
print("Summary:", primeira_pagina.get("summary"))
print("Registros na página 1:", len(primeira_pagina.get("data") or primeira_pagina.get("results") or []))

In [ ]:
registros = (primeira_pagina.get("data") or primeira_pagina.get("results") or [])
pd.DataFrame(registros).head()

### 1.2 Paginação completa (sem gravar ainda)

Itera todas as páginas do intervalo usando `fetch_contratos`, mostrando o progresso página a página — a mesma função usada de verdade na extração, só que aqui coletamos tudo em memória para inspecionar.

In [ ]:
todas_as_paginas = []

for page, records in api_extractor.fetch_contratos(DATA_INICIO, DATA_FIM):
    print(f"Página {page}: {len(records)} registros")
    todas_as_paginas.extend(records)

df_contratos = pd.DataFrame(todas_as_paginas)
print(f"\nTotal de registros: {len(df_contratos)}")
df_contratos.head()

In [ ]:
df_contratos.info()

### 1.3 Extração completa gravando na Bronze

Agora sim usando `extract_and_save`, que grava cada página como JSON no destino configurado em `.env` (`BRONZE_STORAGE_BACKEND=local` ou `hdfs`) e retorna só os metadados (contagens) — o mesmo formato que a task do Airflow vai receber via XCom.

In [ ]:
resultado_api = api_extractor.extract_and_save(DATA_INICIO, DATA_FIM)
print(resultado_api)

## 2. PostgreSQL de origem

Requer `SOURCE_POSTGRES_URL` configurada no `.env` apontando para a infra real do time.

### 2.1 Teste de conexão

In [ ]:
from sqlalchemy import create_engine, text
from src.extractors import postgres_extractor

engine = create_engine(postgres_extractor.SOURCE_DB_URL)

with engine.connect() as conn:
    versao = conn.execute(text("SELECT version()")).scalar()
print("Conectado com sucesso:", versao)

### 2.2 Extração tabela a tabela (sem gravar ainda)

Roda `extract_table` para cada tabela do escopo, mostrando shape e uma amostra — bom momento para conferir se os nomes de coluna de data em `TABLE_DATE_COLUMNS` (`data_empenho`, `data_pagamento`) batem com o schema real.

In [ ]:
DATA_INICIO_PG = "2026-01-01"
DATA_FIM_PG = "2026-01-31"

dataframes = {}
for tabela in postgres_extractor.TABLE_DATE_COLUMNS:
    df = postgres_extractor.extract_table(tabela, DATA_INICIO_PG, DATA_FIM_PG, engine=engine)
    dataframes[tabela] = df
    print(f"{tabela}: shape={df.shape}")

In [ ]:
dataframes["empenhos"].head()

In [ ]:
dataframes["ordem_bancaria_orcamentaria"].head()

In [ ]:
dataframes["unidade_gestora"].head()

### 2.3 Extração completa gravando na Bronze

In [ ]:
resultado_pg = postgres_extractor.extract_and_save(DATA_INICIO_PG, DATA_FIM_PG, engine=engine)
print(resultado_pg)

In [ ]:
engine.dispose()

## Próximos passos

- Conferir se `TABLE_DATE_COLUMNS` em `postgres_extractor.py` bate com o schema real (colunas de data de `empenhos` e `ordem_bancaria_orcamentaria`).
- Confirmar `BRONZE_BASE_PATH` / `HDFS_WEBHDFS_URL` com a infra que o time já tem rodando.
- Fase 2: enriquecimento cruzando `empenhos` × `unidade_gestora`.